# N2 Phase A, iteration 2 (development, Track C)
Runs PROTOCOL_A_ITER2.md (addendum to PROTOCOL_A.md). Refuses to run if the embedded text differs from the signed hash.
ODD attack instances and the development split only; even instances and clean TEST are never computed.
Internet ON (downloads ~7 MB). CPU is enough (about 40-60 min). Run all cells top to bottom.

In [ ]:
import json, hashlib, warnings
warnings.filterwarnings('ignore')
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
PROTOCOL_TEXT = "# N2 Phase A, iteration 2 (of a maximum of 3): addendum to PROTOCOL_A.md\n\nWritten 2026-11-09, AFTER the iteration-1 results were seen. Every change below is a change made in response to a shown failure and is logged as such. Phase A remains development: nothing here is confirmed. Phase B will fix all rules in a new signed pre-registration before the even-numbered instances or the clean TEST days are touched.\n\n## 1. What iteration 1 showed (files read: detection_dev_all.csv, identity_runs.json)\n- Detection, autoencoder (AE), static threshold, per-device expert E: mean DEV-VAL alarm rate 0.066, but without NetatmoCamera 0.012 (detection 0.895). NetatmoCamera alarm rate was 0.499 for E and 0.500 for G1: the threshold taken from CALIB days does not transfer to its DEV-VAL days (drift). The same device failed the same way in the first N2 run.\n- Rolling median+k*MAD thresholds fixed NetatmoCamera (alarm 0.000, detection 1.0) but made the other devices worse (mean alarm 0.079 without Netatmo; k reached its cap of 30 for HueBulb), so that rule is dropped.\n- The configuration-level admissibility rule (mean alarm over devices <=0.02) let one drifting device veto a configuration that is clean on the other devices. The device-level tiering already handles per-device failure.\n- Correction: my earlier suggestion that counting false alarms per minute versus per 2-of-N block was the main problem was not supported. Block-level alarm rates are only reported, not used.\n- Identity: every device has F1 = 1.0 (separable). Claim acceptance with the CALIB-only threshold was low for SamsungCamera (0.22 vote3, 0.29 single), HueBulb 0.78, iHome 0.63. Cause not yet established.\n- IsolationForest (no admissible configuration), the lag-stacked autoencoder (never better than AE), the MLP identity arm (not admitted) and the supervised leave-one-family-out reference (0.631, worse than AE) are closed. Not rerun.\n\n## 2. Detection changes (D1-D3)\nD1 Family: AE only (same architecture, training caps, seeds 11/22/33, scopes E, G0, G1 as in PROTOCOL_A section 4). Modes:\n- static: unchanged (99th percentile of CALIB scores; G0 pooled).\n- rollq: threshold at minute t = 99th percentile of that device's own scores over the preceding 4,320 rows (past only, shifted by one, at least 1,440 rows, otherwise the static threshold). Scope G0 uses the pooled static threshold as fallback.\n- both: alarm only when both the static and the rollq threshold are exceeded. Why: static alarms on drifted devices and rollq has lower alarms there; requiring both limits false alarms from either failure.\nD2 Selection (replaces the ladder of PROTOCOL_A section 5 for this family): a configuration (mode, scope) is scored by the number of eligible devices it SUPPORTS = seed-mean DEV detection >=0.80 AND seed-mean DEV-VAL minute alarm rate <=0.02. Selected = most supported devices; ties: higher mean detection over supported devices, then simpler (static before rollq before both; G0 before G1 before E).\nRouting test (replaces PROTOCOL_A section 6 for this family): per mode, E versus the better global scope (chosen the same way) on the COMMON CLEAN set = eligible devices where both have alarm <=0.02. Passes if macro DEV detection difference >=+0.05 in every seed and the paired bootstrap (2,000 resamples, seed 20261108, seed-averaged hits) has lower 95% bound >0. Routing is KEPT only if the selected scope is E and its mode passes. Otherwise routing is removed and the best global configuration (scopes G0/G1 only) is the detector.\nD3 Per-minute scores are saved for diagnostics. No further mode may be evaluated from them without a logged iteration.\nReported only: block false-alarm rate (non-overlapping 10-minute blocks of DEV-VAL with >=2 alarm minutes), CALIB alarm rate.\nDevice detection support for tiers is unchanged: seed-mean DEV detection >=0.80, DEV-VAL alarm <=0.02, >=4 DEV instances.\n\n## 3. Identity changes (I1)\nArms: RF(500) single and vote3 only (XGBoost dropped: never primary; MLP dropped: not admitted). Seeds 11/22/33. Held-out run and closed-set run as before.\nThreshold rules: T0 = 10th percentile of the true-claim probability on CALIB windows (iteration 1). T1 = 10th percentile of the pooled true-claim probabilities of CALIB windows and day-grouped 5-fold out-of-fold DEV-TRAIN windows (RF with 300 trees per fold). T1 uses no DEV-VAL data.\nPrimary arm: higher seed-mean CALIB macro-F1 among the two (T0 value used).\nAdoption of T1: only if (a) with T1 the held-out run still meets G2>=0.80, G3>=0.95, G4>=0.90, G5>=0.60, G6<=0.10, (b) the number of closed-set identity-supported devices strictly increases, and (c) G6 under T1 is at most G6 under T0 + 0.01. Otherwise T0 stays.\nIdentity support per device unchanged: acceptance >=0.80, F1 >=0.80, claim rejection >=0.95.\nDiagnostics reported: per device median true-claim probability on OOF, CALIB and DEV-VAL windows and acceptance on DEV-VAL under T0 and T1.\n\n## 4. Unchanged\nOdd/even rule, development split, features, tier definitions, state definitions, the rule that even-instance outcomes and clean TEST are never computed.\n\n## 5. Next\nAfter this iteration the tiers, algorithms and rules are frozen in the Phase B pre-registration. Iteration 3 is used only if this run shows a clear, specific failure that is logged first. The disclosure about A1 having seen IsolationForest outcomes on even instances stands.\n"
CHAIN_BLOCK = "{\"hash\": \"4b23307ef13e90290663b29830d69ed9c34e0c1f1f095516221c1ff481e53715\", \"header\": {\"index\": 3, \"merkle_root\": \"da4fa4d9d62da2e8736fd3b1579f6fe732f8889c0305ac2ec6e610cda5fc1964\", \"prev_hash\": \"659469f4a5a5070b210813fe6af797eb90899a0787dfafaed6c2733891f5ea0f\", \"signer_pub\": \"93d2db7c37ec10acb6e189b61f6a5eacdddb74afd482d9e5bbcd3c41e1213b23\", \"timestamp\": 1794165558.4608498}, \"payload\": {\"experiment\": \"N2\", \"file\": \"PROTOCOL_A_ITER2.md\", \"iteration\": 2, \"kind\": \"protocol_freeze_iter2\", \"note\": \"iteration 2 addendum; written after iteration-1 results; signed before the iteration-2 notebook is run\", \"phase\": \"A\", \"sha256\": \"8ff35c398f2a577499be4615c5f3e3d4de1913638ece273ef74d2a4907881b12\"}, \"sig\": \"84312325e34c9f96b199662bfcf7aca6517b654f84ef038e6330e14b2fe2f0ef9a58d5b5a17cc2c01a0057df59c7dc6b2944902fe0fd076ffa1a1278e5138602\"}"
PUB = '93d2db7c37ec10acb6e189b61f6a5eacdddb74afd482d9e5bbcd3c41e1213b23'
def _canon(o): return json.dumps(o, sort_keys=True, separators=(',', ':')).encode()
blk = json.loads(CHAIN_BLOCK)
sha = hashlib.sha256(PROTOCOL_TEXT.encode()).hexdigest()
assert blk['payload']['sha256'] == sha, 'PROTOCOL_A_ITER2 text differs from the signed version: STOP'
assert blk['payload']['kind'] == 'protocol_freeze_iter2' and blk['header']['signer_pub'] == PUB
assert hashlib.sha256(_canon({'header': blk['header'], 'payload': blk['payload']})).hexdigest() == blk['hash'], 'block hash mismatch'
Ed25519PublicKey.from_public_bytes(bytes.fromhex(PUB)).verify(bytes.fromhex(blk['sig']), blk['hash'].encode())
print('PROTOCOL_A_ITER2 signature OK | sha256', sha, '| chain block', blk['header']['index'], blk['hash'])

## Libraries (embedded; same files that passed the synthetic smoke test)

In [ ]:
"""N2 core: identity + per-device attack detection + routing on the UNSW attack dataset.
Implements PREREG_N2.md exactly. Interpretations the prereg left open are marked INTERPRETATION."""
import os, re, json, glob, hashlib
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

DEVICES = {'ec1a59832811': ('WeMoMotion', 1), 'ec1a5979f489': ('WeMoSwitch', 1), '00166cab6b88': ('SamsungCamera', 1),
           '50c7bf005639': ('TPLinkPlug', 1), '70ee50183443': ('NetatmoCamera', 1),
           '0017882b9a25': ('HueBulb', 2), '44650d56ccd3': ('Echo', 2), 'f4f5d88f0a3c': ('Chromecast', 2),
           '74c63b29d71d': ('iHome', 2), 'd073d5018308': ('LIFX', 2)}
CFG = dict(split_seed=20261108, seeds=(11, 22, 33), if_trees=300, rf_trees=500, xgb_trees=500,
           boot=1000, boot_inst=2000, min_attack_inst=5)
COL_RE = re.compile(r'^(From|To)(Local|Internet)(.+?)(Packet|Byte)(?:\.\d+)?$')
FEATS = [f'{d}{l}_{p}_{u}' for d in ('From', 'To') for l in ('Local', 'Internet')
         for p in ('Tcp', 'Udp', 'Icmp', 'Arp', 'Other') for u in ('Packet', 'Byte')]

def proto_class(mid):
    for k in ('Tcp', 'Udp', 'Icmp', 'Arp'):
        if mid.startswith(k): return k
    return 'Other'

def common_features(df):
    groups, unmatched = {}, []
    for c in df.columns:
        if c == 'Timestamp': continue
        m = COL_RE.match(str(c))
        if not m: unmatched.append(c); continue
        d, l, mid, u = m.groups()
        groups.setdefault(f'{d}{l}_{proto_class(mid)}_{u}', []).append(c)
    X = pd.DataFrame(0.0, index=df.index, columns=FEATS)
    for k, cl in groups.items():
        X[k] = df[cl].apply(pd.to_numeric, errors='coerce').fillna(0).clip(lower=0).sum(axis=1)
    other_cols = sum(len(v) for k, v in groups.items() if '_Other_' in k)
    return np.log1p(X), unmatched, other_cols

def minute_table(df):
    df = df.sort_values('Timestamp').reset_index(drop=True)
    X, unm, oc = common_features(df)
    M = X.rolling(5, min_periods=1).mean()
    Xn = X.to_numpy(np.float32)
    return dict(ts=df['Timestamp'].astype('int64').to_numpy(), X=Xn,
                F=np.hstack([Xn, M.to_numpy(np.float32)]), unmatched=unm, other_cols=oc)

ATK_RE = re.compile(r'^(.*?)(\d+)([LWD]2[LWD])$')
def parse_annotations(path):
    inst = []
    for line in open(path, errors='replace'):
        line = line.strip()
        if not line: continue
        h = line.split(',', 2)
        if len(h) < 3: raise ValueError(f'unparseable annotation line in {path}: {line[:80]}')
        s, e = float(h[0]), float(h[1])            # raises if a header sneaks in: STOP rule
        name = h[2].rpartition(',')[2].strip()
        inst.append((s, e, name))
    inst = sorted(set(inst))
    if not inst: raise ValueError(f'no instances in {path}')
    return inst

def attack_parts(name):
    m = ATK_RE.match(name)
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (name, -1, '')

def split_days(ts, inst, min_rows=600):
    day = (ts // 86400000).astype(int)
    neigh = np.zeros(len(ts), bool)
    for s, e, _ in inst:
        neigh |= (ts >= (s - 7200) * 1000) & (ts <= (e + 7200) * 1000)
    ad = set(day[neigh].tolist()); adb = ad | {d - 1 for d in ad} | {d + 1 for d in ad}
    u, c = np.unique(day, return_counts=True)
    clean = [int(d) for d, k in zip(u, c) if k >= min_rows and int(d) not in adb]
    n = len(clean)
    if n < 10: raise ValueError(f'only {n} clean days')
    a = int(0.70 * n); b = a + 1 + int(0.15 * n)
    return day, neigh, dict(train=set(clean[:a]), calib=set(clean[a + 1:b]), test=set(clean[b + 1:])), n

def load_all(flow_dir, ann_dir):
    D = {}
    for mac, (name, coh) in DEVICES.items():
        fp = glob.glob(f'{flow_dir}/**/{mac}_flowstats.csv', recursive=True)
        ap = glob.glob(f'{ann_dir}/**/{mac}.csv', recursive=True)
        if not fp or not ap: raise FileNotFoundError(f'{mac}: flow={bool(fp)} annotation={bool(ap)}')
        mt = minute_table(pd.read_csv(fp[0], low_memory=False))
        inst = parse_annotations(ap[0])
        day, neigh, sp, n = split_days(mt['ts'], inst)
        d = dict(name=name, cohort=coh, inst=inst, day=day, neigh=neigh, split=sp, n_clean=n, **mt)
        for k in ('train', 'calib', 'test'):
            d['m_' + k] = np.isin(day, list(sp[k])) & ~neigh
        D[mac] = d
    return D

# ---------------- detection ----------------
def fit_if(X, seeds, trees):
    return [IsolationForest(n_estimators=trees, random_state=s, n_jobs=-1).fit(X) for s in seeds]
def score_if(models, X):
    return np.mean([-m.score_samples(X) for m in models], axis=0)
def inst_hits(ts, sc, thr, inst, min_alarms=2):
    return np.array([int(((sc[(ts >= s * 1000) & (ts <= e * 1000)]) >= thr).sum() >= min_alarms) for s, e, _ in inst])

def run_detection(D, cfg):
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}
    oh = lambda m, n: np.tile(np.eye(len(macs), dtype=np.float32)[idx[m]], (n, 1))
    E = {m: fit_if(D[m]['F'][D[m]['m_train']], cfg['seeds'], cfg['if_trees']) for m in macs}
    G0 = fit_if(np.vstack([D[m]['F'][D[m]['m_train']] for m in macs]), cfg['seeds'], cfg['if_trees'])
    G1 = fit_if(np.vstack([np.hstack([D[m]['F'][D[m]['m_train']], oh(m, int(D[m]['m_train'].sum()))]) for m in macs]),
                cfg['seeds'], cfg['if_trees'])
    sE, s0, s1 = {}, {}, {}
    for m in macs:
        F = D[m]['F']; n = len(F)
        sE[m] = score_if(E[m], F); s0[m] = score_if(G0, F); s1[m] = score_if(G1, np.hstack([F, oh(m, n)]))
    thrE = {m: np.quantile(sE[m][D[m]['m_calib']], 0.99) for m in macs}
    thr0 = np.quantile(np.concatenate([s0[m][D[m]['m_calib']] for m in macs]), 0.99)
    thr1 = {m: np.quantile(s1[m][D[m]['m_calib']], 0.99) for m in macs}
    rows, per_inst = [], []
    for m in macs:
        d = D[m]; ts = d['ts']; te = d['m_test']
        hE = inst_hits(ts, sE[m], thrE[m], d['inst']); h0 = inst_hits(ts, s0[m], thr0, d['inst']); h1 = inst_hits(ts, s1[m], thr1[m], d['inst'])
        rows.append(dict(mac=m, device=d['name'], cohort=d['cohort'], n_inst=len(d['inst']), eligible=len(d['inst']) >= cfg['min_attack_inst'],
                         det_E=hE.mean(), det_G0=h0.mean(), det_G1=h1.mean(),
                         alarm_E=float((sE[m][te] >= thrE[m]).mean()), alarm_G0=float((s0[m][te] >= thr0).mean()), alarm_G1=float((s1[m][te] >= thr1[m]).mean())))
        for k, (s, e, nm) in enumerate(d['inst']):
            f, r, p = attack_parts(nm)
            per_inst.append(dict(mac=m, device=d['name'], name=nm, family=f, rate=r, path=p, hit_E=int(hE[k]), hit_G0=int(h0[k]), hit_G1=int(h1[k]),
                                 n_rows=int(((ts >= s * 1000) & (ts <= e * 1000)).sum())))
    det = pd.DataFrame(rows); pi = pd.DataFrame(per_inst)
    # wrong-expert: expert of A applied to device B's attack traffic
    wrong = []
    for B in macs:
        dB = D[B]
        for A in macs:
            if A == B: continue
            sc = score_if(E[A], dB['F'])
            wrong.append(dict(claimed=D[A]['name'], actual=dB['name'], same_cohort=D[A]['cohort'] == dB['cohort'],
                              det=float(inst_hits(dB['ts'], sc, thrE[A], dB['inst']).mean()),
                              eligible=len(dB['inst']) >= cfg['min_attack_inst']))
    return det, pi, pd.DataFrame(wrong)

def macro(det, col, elig=True):
    t = det[det.eligible] if elig else det
    return float(t[col].mean())

def routing_verdict(det, pi, cfg, seed=20261108):
    el = det[det.eligible]
    mE, m0, m1 = macro(det, 'det_E'), macro(det, 'det_G0'), macro(det, 'det_G1')
    best = 'G0' if m0 >= m1 else 'G1'; mG = max(m0, m1)
    aE = macro(det, 'alarm_E'); aG = macro(det, 'alarm_' + best)
    rng = np.random.default_rng(seed); macs = el.mac.tolist()
    hits = {m: pi[pi.mac == m][['hit_E', 'hit_' + best]].to_numpy() for m in macs}
    diffs = []
    for _ in range(cfg['boot_inst']):
        dd = []
        for m in macs:
            h = hits[m]; i = rng.integers(0, len(h), len(h)); dd.append(h[i, 0].mean() - h[i, 1].mean())
        diffs.append(np.mean(dd))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    accepted = bool((mE - mG) >= 0.05 and lo > 0 and aE <= aG + 0.005)
    gate_A1 = bool(mE >= 0.80 and aE <= 0.02)
    return dict(det_E=mE, det_G0=m0, det_G1=m1, best_global=best, delta=mE - mG, delta_ci95=[float(lo), float(hi)],
                alarm_E=aE, alarm_best_global=aG, gate_A1_expert=gate_A1,
                routing='routing_accepted' if accepted else 'routing_rejected_use_global', eligible_devices=el.device.tolist())

# ---------------- identity ----------------
def windows(d, days, win_ms=15 * 60000, min_rows=10):
    sel = np.isin(d['day'], list(days)) & ~d['neigh']
    if not sel.any(): return np.zeros((0, 120), np.float32), np.zeros(0, int)
    t = d['ts'][sel]; x = d['X'][sel]; df = pd.DataFrame(x); df['w'] = t // win_ms; df['day'] = d['day'][sel]
    g = df.groupby('w'); keep = g.size(); keep = keep[keep >= min_rows].index
    cols = list(range(x.shape[1]))
    agg = g[cols].agg(['mean', 'std', 'max']).loc[keep]
    F = agg.to_numpy(np.float32); F = np.nan_to_num(F)
    return F, g['day'].first().loc[keep].to_numpy()

def vote(P, k=3):
    if len(P) == 0: return P
    return pd.DataFrame(P).rolling(k, min_periods=1).mean().to_numpy()

def make_model(kind, cfg, n_classes):
    if kind == 'RF':
        return RandomForestClassifier(n_estimators=cfg['rf_trees'], class_weight='balanced', n_jobs=-1, random_state=11)
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=cfg['xgb_trees'], max_depth=6, learning_rate=0.05, objective='multi:softprob',
                         tree_method='hist', n_jobs=-1, random_state=11, num_class=n_classes)

def pick_unknown(D, seed):
    rng = np.random.default_rng(seed); out = []
    for coh in (1, 2):
        ms = sorted(m for m in D if D[m]['cohort'] == coh); out.append(ms[int(rng.integers(0, len(ms)))])
    return out

def cluster_boot(keys, vals, B, seed):
    keys = np.asarray(keys); vals = np.asarray(vals, float)
    u, inv = np.unique(keys, return_inverse=True)
    s = np.bincount(inv, vals); c = np.bincount(inv).astype(float)
    rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        i = rng.integers(0, len(u), len(u)); out.append(s[i].sum() / c[i].sum())
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]

def state_of(P_row, claim, thr, known_idx):
    if claim is None: return 'UNREGISTERED_CLAIM'
    if P_row[claim] >= thr[claim]: return 'MATCH'
    pm = P_row.max()
    return 'MISMATCH' if pm >= 0.60 else ('UNKNOWN' if pm < 0.35 else 'UNCERTAIN')

def run_identity(D, cfg):
    unk = pick_unknown(D, cfg['split_seed']); known = sorted(m for m in D if m not in unk)
    kidx = {m: i for i, m in enumerate(known)}; K = len(known)
    W = {}
    for m in D:
        for part in ('train', 'calib', 'test'):
            W[(m, part)] = windows(D[m], D[m]['split'][part])
            if len(W[(m, part)][0]) == 0:   # STOP rule: report, do not silently drop a device
                raise ValueError(f'no identity windows for {D[m]["name"]} in {part}')
    def stack(part, devs):
        Xs, ys, ds, dv = [], [], [], []
        for m in devs:
            F, dy = W[(m, part)]; Xs.append(F); ys += [kidx.get(m, -1)] * len(F); ds += list(dy); dv += [m] * len(F)
        return np.vstack(Xs), np.array(ys), np.array(ds), np.array(dv)
    Xtr, ytr, _, _ = stack('train', known); Xca, yca, dca, vca = stack('calib', known); Xte, yte, dte, vte = stack('test', known)
    Xun, _, dun, vun = stack('test', unk)
    arms = {}
    for kind in ('RF', 'XGB'):
        mdl = make_model(kind, cfg, K)
        if kind == 'XGB': mdl.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
        else: mdl.fit(Xtr, ytr)
        P = {n: mdl.predict_proba(X) for n, X in (('ca', Xca), ('te', Xte), ('un', Xun))}
        def per_dev_vote(P_, vv):
            out = P_.copy()
            for m in np.unique(vv):
                ii = np.where(vv == m)[0]; out[ii] = vote(P_[ii])
            return out
        for mode in ('single', 'vote3'):
            Q = P if mode == 'single' else {'ca': per_dev_vote(P['ca'], vca), 'te': per_dev_vote(P['te'], vte), 'un': per_dev_vote(P['un'], vun)}
            arms[f'{kind}-{mode}'] = dict(P=Q, f1_calib=float(f1_score(yca, Q['ca'].argmax(1), average='macro')))
    order = ['RF-single', 'RF-vote3', 'XGB-single', 'XGB-vote3']
    primary = max(order, key=lambda a: (round(arms[a]['f1_calib'], 12), -order.index(a)))
    Pa = arms[primary]['P']
    thr = np.array([np.quantile(Pa['ca'][yca == k, k], 0.10) for k in range(K)])
    pred = Pa['te'].argmax(1); res = dict(primary=primary, calib_macro_f1={a: arms[a]['f1_calib'] for a in order}, unknown_devices=unk, known_devices=known)
    G1 = float(f1_score(yte, pred, average='macro'))
    acc = (Pa['te'][np.arange(len(yte)), yte] >= thr[yte]).astype(float)
    per_dev_acc = {known[k]: float(acc[yte == k].mean()) for k in range(K)}
    G2 = float(np.mean(list(per_dev_acc.values())))              # INTERPRETATION: macro over known devices
    rng = np.random.default_rng(cfg['split_seed'])
    claim_r = np.array([rng.choice([k for k in range(K) if k != y]) for y in yte])
    rej_r = (Pa['te'][np.arange(len(yte)), claim_r] < thr[claim_r]).astype(float)
    hard = {}
    for A in range(K):
        mp = {B: Pa['ca'][yca == B][:, A].mean() for B in range(K) if B != A}; hard[A] = max(mp, key=mp.get)
    rows_h = [(i, A) for A in range(K) for i in np.where(yte == hard[A])[0]]
    rej_h = np.array([float(Pa['te'][i, A] < thr[A]) for i, A in rows_h]) if rows_h else np.array([1.0])
    key_h = [f'{vte[i]}_{dte[i]}' for i, A in rows_h]
    rej_u = np.array([[float(Pa['un'][i, A] < thr[A]) for A in range(K)] for i in range(len(Xun))]) if len(Xun) else np.zeros((1, K))
    rej_u_flat = rej_u.ravel(); key_u = np.repeat([f'{vun[i]}_{dun[i]}' for i in range(len(Xun))], K) if len(Xun) else ['x']
    G3 = float(rej_r.mean()); G4 = float(rej_h.mean()); G5 = float(rej_u_flat.mean())
    G6 = float(1 - np.concatenate([rej_r, rej_h, rej_u_flat]).mean())
    B = cfg['boot']; kt = [f'{vte[i]}_{dte[i]}' for i in range(len(yte))]
    ci = dict(G1=cluster_boot(kt, (pred == yte).astype(float), B, 1)[:], G2=cluster_boot(kt, acc, B, 2), G3=cluster_boot(kt, rej_r, B, 3),
              G4=cluster_boot(key_h, rej_h, B, 4), G5=cluster_boot(key_u, rej_u_flat, B, 5))
    ci['G1'] = 'accuracy CI shown (macro-F1 CI not bootstrapped)'; ci['note'] = 'cluster bootstrap over device-days'
    gates = dict(G1=G1 >= 0.80, G2=G2 >= 0.80, G3=G3 >= 0.95, G4=G4 >= 0.90, G5=G5 >= 0.60, G6=G6 <= 0.10)
    nf = sum(not v for k, v in gates.items() if k != 'G1')
    verdict = 'POSITIVE' if all(gates.values()) else ('PARTIAL' if gates['G1'] and nf <= 1 else 'NEGATIVE')
    # leaky comparison
    allX = np.vstack([Xtr, Xca, Xte]); ally = np.concatenate([ytr, yca, yte]); fl = []
    for a, b in StratifiedKFold(5, shuffle=True, random_state=11).split(allX, ally):
        mdl = make_model(primary.split('-')[0], cfg, K)
        (mdl.fit(allX[a], ally[a], sample_weight=compute_sample_weight('balanced', ally[a])) if primary.startswith('XGB') else mdl.fit(allX[a], ally[a]))
        fl.append(f1_score(ally[b], mdl.predict(allX[b]), average='macro'))
    # states
    def states(P_rows, claims):
        return pd.Series([state_of(P_rows[i], claims[i], thr, kidx) for i in range(len(P_rows))]).value_counts(normalize=True).round(4).to_dict()
    S = dict(honest=states(Pa['te'], list(yte)), random_mismatch=states(Pa['te'], list(claim_r)),
             hard_mismatch=states(Pa['te'][[i for i, A in rows_h]], [A for i, A in rows_h]) if rows_h else {},
             unknown_device=states(np.repeat(Pa['un'], K, axis=0), list(np.tile(np.arange(K), len(Xun)))) if len(Xun) else {})
    confusion = pd.crosstab(pd.Series([known[i] for i in yte], name='actual'), pd.Series([known[i] for i in pred], name='pred'))
    res.update(G1_macro_f1=G1, G2_honest_acceptance_macro=G2, G2_micro=float(acc.mean()), G3_random_mismatch_rejection=G3,
               G4_hard_mismatch_rejection=G4, G5_unknown_rejection=G5, G6_false_verification=G6, gates=gates, verdict=verdict,
               per_device_acceptance=per_dev_acc, hard_partner={known[A]: known[hard[A]] for A in range(K)}, ci95=ci,
               leaky_random_5fold_macro_f1=float(np.mean(fl)), states=S, n_windows=dict(train=len(Xtr), calib=len(Xca), test=len(Xte), unknown_test=len(Xun)))
    return res, confusion

In [ ]:
"""N2 Phase A (development). Implements PROTOCOL_A.md. Requires n2_lib.py (same notebook) for loading and features.
Outcomes on EVEN-numbered attack instances and on clean TEST days are never computed here (d['dev_inst'] holds the ODD ones only)."""
import warnings
import numpy as np, pandas as pd
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
try:                                   # in the notebook these names already exist (n2_lib is pasted above)
    from n2_lib import attack_parts, windows, vote
except ImportError:
    pass

ACFG = dict(seeds=(11, 22, 33), ae_epochs=60, ae_hidden=(64, 16, 64), tae_hidden=(128, 32, 128), tae_lags=10,
            cap_dev=40000, cap_pool=150000, min_dev_inst=4, clip=20.0,
            roll_win=4320, roll_min=1440, k_grid=(1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 30),
            alarm_adm=0.02, det_margin=0.05, alarm_slack=0.005, boot_inst=2000, boot_seed=20261108)
FAMILIES = ('IF', 'AE', 'TAE'); MODES = ('static', 'rolling'); SCOPES = ('G0', 'G1', 'E')

# ---------------- development split ----------------
def prepare_dev(D, acfg=ACFG):
    for m, d in D.items():
        tr = sorted(d['split']['train']); a = int(0.75 * len(tr))
        dtr, dval = set(tr[:a]), set(tr[a + 1:])
        d['dev_days'] = (dtr, dval)
        d['m_dtrain'] = np.isin(d['day'], list(dtr)) & ~d['neigh']
        d['m_dval'] = np.isin(d['day'], list(dval)) & ~d['neigh']
        d['dev_inst'] = d['inst'][0::2]                     # ODD instances (1st, 3rd, ...) = DEV
        d['eligible'] = len(d['dev_inst']) >= acfg['min_dev_inst']
        if d['m_dtrain'].sum() < 500 or d['m_dval'].sum() < 100:
            raise ValueError(f"{d['name']}: development split too small (train {d['m_dtrain'].sum()}, val {d['m_dval'].sum()})")
        d.pop('m_test', None)                                 # clean TEST mask removed: cannot be used by mistake
    return D

# ---------------- detectors ----------------
def lagstack(X, L):
    n = len(X); out = np.empty((n, X.shape[1] * L), np.float32)
    for j in range(L):
        idx = np.maximum(np.arange(n) - j, 0)
        out[:, j * X.shape[1]:(j + 1) * X.shape[1]] = X[idx]
    return out

class AE:
    def __init__(self, hidden, epochs, seed, clip):
        self.h, self.ep, self.seed, self.clip = hidden, epochs, seed, clip
    def _z(self, X): return np.clip(self.sc.transform(X), -self.clip, self.clip)
    def fit(self, X):
        self.sc = StandardScaler().fit(X); Z = self._z(X)
        self.m = MLPRegressor(hidden_layer_sizes=self.h, activation='relu', solver='adam', max_iter=self.ep, batch_size=256,
                              learning_rate_init=1e-3, tol=1e-6, n_iter_no_change=15, random_state=self.seed).fit(Z, Z)
        return self
    def score(self, X):
        out = np.empty(len(X))
        for i in range(0, len(X), 50000):
            Z = self._z(X[i:i + 50000]); out[i:i + 50000] = ((self.m.predict(Z) - Z) ** 2).mean(1)
        return out

class IFW:
    def __init__(self, trees, seed): self.t, self.s = trees, seed
    def fit(self, X):
        from sklearn.ensemble import IsolationForest
        self.m = IsolationForest(n_estimators=self.t, random_state=self.s, n_jobs=-1).fit(X); return self
    def score(self, X): return -self.m.score_samples(X)

def rolling_thr(s, calib, static_thr, acfg):
    ser = pd.Series(s)
    med = ser.rolling(acfg['roll_win'], min_periods=acfg['roll_min']).median().shift(1)
    mad = (ser - med).abs().rolling(acfg['roll_win'], min_periods=acfg['roll_min']).median().shift(1)
    best = None
    for k in acfg['k_grid']:
        thr = (med + k * 1.4826 * mad).to_numpy(); thr = np.where(np.isnan(thr), static_thr, thr)
        best = (k, thr)
        if (s[calib] >= thr[calib]).mean() <= 0.01: break
    return best

def hits_arr(ts, alarm, inst):
    out = []
    for s, e, _ in inst:
        lo = np.searchsorted(ts, s * 1000, 'left'); hi = np.searchsorted(ts, e * 1000, 'right')
        out.append(int(alarm[lo:hi].sum() >= 2))
    return np.array(out, int)

def run_family(D, family, seed, acfg=ACFG, cfg=None):
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}; eye = np.eye(len(macs), dtype=np.float32)
    rng = np.random.default_rng(seed)
    def Xin(m):
        d = D[m]; return d['F'] if family != 'TAE' else lagstack(d['X'], acfg['tae_lags'])
    def new():
        if family == 'IF': return IFW(cfg['if_trees'], seed)
        return AE(acfg['ae_hidden'] if family == 'AE' else acfg['tae_hidden'], acfg['ae_epochs'], seed, acfg['clip'])
    def samp(m, n):
        ii = np.where(D[m]['m_dtrain'])[0]
        if family != 'IF' and len(ii) > n: ii = np.sort(rng.choice(ii, n, replace=False))
        return ii
    X = {m: Xin(m) for m in macs}
    E = {m: new().fit(X[m][samp(m, acfg['cap_dev'])]) for m in macs}
    per = acfg['cap_pool'] // len(macs)
    Xp = {m: X[m][samp(m, per)] for m in macs}
    G0 = new().fit(np.vstack([Xp[m] for m in macs]))
    G1 = new().fit(np.vstack([np.hstack([Xp[m], np.tile(eye[idx[m]], (len(Xp[m]), 1))]) for m in macs]))
    S = {'E': {}, 'G0': {}, 'G1': {}}
    for m in macs:
        S['E'][m] = E[m].score(X[m]); S['G0'][m] = G0.score(X[m])
        S['G1'][m] = G1.score(np.hstack([X[m], np.tile(eye[idx[m]], (len(X[m]), 1))]))
    del X, Xp
    thr0 = np.quantile(np.concatenate([S['G0'][m][D[m]['m_calib']] for m in macs]), 0.99)
    rows, prow = [], []
    for scope in SCOPES:
        for m in macs:
            d = D[m]; s = S[scope][m]
            st = thr0 if scope == 'G0' else np.quantile(s[d['m_calib']], 0.99)
            rk, rthr = rolling_thr(s, d['m_calib'], st, acfg)
            for mode, thr, k in (('static', np.full(len(s), st), np.nan), ('rolling', rthr, rk)):
                al = s >= thr; h = hits_arr(d['ts'], al, d['dev_inst'])
                rows.append(dict(family=family, mode=mode, scope=scope, seed=seed, mac=m, device=d['name'], eligible=d['eligible'],
                                 n_dev_inst=len(d['dev_inst']), det=float(h.mean()), alarm_val=float(al[d['m_dval']].mean()), k=k))
                for j, v in enumerate(h): prow.append((family, mode, scope, seed, m, j, int(v)))
    pi = pd.DataFrame(prow, columns=['family', 'mode', 'scope', 'seed', 'mac', 'inst', 'hit'])
    return pd.DataFrame(rows), pi

# ---------------- supervised reference (leave one attack family out) ----------------
def run_supervised(D, seed, acfg=ACFG, cfg=None):
    from xgboost import XGBClassifier
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}; eye = np.eye(len(macs), dtype=np.float32)
    rng = np.random.default_rng(seed)
    def oh(m, n): return np.tile(eye[idx[m]], (n, 1))
    ben = {}
    for m in macs:
        ii = np.where(D[m]['m_dtrain'])[0]
        if len(ii) > acfg['cap_pool'] // len(macs): ii = np.sort(rng.choice(ii, acfg['cap_pool'] // len(macs), replace=False))
        ben[m] = ii
    atk = []                                                     # (mac, family, row indices)
    for m in macs:
        d = D[m]
        for s, e, nm in d['dev_inst']:
            lo = np.searchsorted(d['ts'], s * 1000, 'left'); hi = np.searchsorted(d['ts'], e * 1000, 'right')
            atk.append((m, attack_parts(nm)[0], np.arange(lo, hi), nm))
    fams = sorted({a[1] for a in atk}); out, alarms = [], {m: [] for m in macs}
    if len(fams) < 2: return pd.DataFrame(), 'single family: leave-one-family-out undefined'
    for f in fams:
        Xb = np.vstack([np.hstack([D[m]['F'][ben[m]], oh(m, len(ben[m]))]) for m in macs]); yb = np.zeros(len(Xb))
        tr = [a for a in atk if a[1] != f and len(a[2])]
        if not tr: continue
        Xa = np.vstack([np.hstack([D[a[0]]['F'][a[2]], oh(a[0], len(a[2]))]) for a in tr]); ya = np.ones(len(Xa))
        mdl = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method='hist', n_jobs=-1, random_state=seed)
        w = np.concatenate([np.ones(len(Xb)), np.full(len(Xa), max(1.0, len(Xb) / max(len(Xa), 1) / 5))])
        mdl.fit(np.vstack([Xb, Xa]), np.concatenate([yb, ya]), sample_weight=w)
        for m in macs:
            d = D[m]; p = mdl.predict_proba(np.hstack([d['F'], oh(m, len(d['F']))]))[:, 1]
            thr = max(np.quantile(p[d['m_calib']], 0.99), 0.5); al = p >= thr; alarms[m].append(float(al[d['m_dval']].mean()))
            for a in [a for a in atk if a[0] == m and a[1] == f]:
                out.append(dict(mac=m, device=d['name'], family=f, inst=a[3], hit=int(al[a[2]].sum() >= 2)))
    r = pd.DataFrame(out)
    summ = r.groupby(['mac', 'device']).hit.agg(['mean', 'count']).reset_index().rename(columns={'mean': 'det_lofo', 'count': 'n_inst'})
    summ['alarm_val'] = summ.mac.map({m: float(np.mean(v)) if v else np.nan for m, v in alarms.items()})
    summ['eligible'] = summ.mac.map({m: D[m]['eligible'] for m in macs})
    return summ, r

# ---------------- selection ladder and routing ----------------
def cfg_table(dev):
    el = dev[dev.eligible]
    g = el.groupby(['family', 'mode', 'scope', 'seed']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean')).reset_index()
    rows = []
    for (f, m, s), t in g.groupby(['family', 'mode', 'scope']):
        t = t.sort_values('seed')
        rows.append(dict(family=f, mode=m, scope=s, det_by_seed=t.det.tolist(), alarm_by_seed=t.alarm.tolist(),
                         det_mean=float(t.det.mean()), alarm_mean=float(t.alarm.mean()), admissible=bool(t.alarm.mean() <= ACFG['alarm_adm'])))
    return pd.DataFrame(rows)

def _best(ct, family, scopes):
    t = ct[(ct.family == family) & ct.admissible & ct.scope.isin(scopes)].copy()
    if t.empty: return None
    t['mr'] = t['mode'].map({'static': 0, 'rolling': 1}); t['sr'] = t.scope.map({'G0': 0, 'G1': 1, 'E': 2}); t['d'] = t.det_mean.round(12)
    return t.sort_values(['d', 'mr', 'sr'], ascending=[False, True, True]).iloc[0]

def ladder(ct, scopes=SCOPES, acfg=ACFG):
    inc, log = None, []
    for f in FAMILIES:
        b = _best(ct, f, scopes)
        if b is None: log.append(f'{f}: no admissible configuration'); continue
        if inc is None:
            inc = b; log.append(f"{f}: incumbent {b['mode']}/{b.scope} det {b.det_mean:.3f} alarm {b.alarm_mean:.4f}"); continue
        gain = [c - i for c, i in zip(b.det_by_seed, inc.det_by_seed)]
        win = all(g >= acfg['det_margin'] for g in gain) and b.alarm_mean <= inc.alarm_mean + acfg['alarm_slack']
        log.append(f"{f}: best {b['mode']}/{b.scope} det {b.det_mean:.3f} alarm {b.alarm_mean:.4f}; gains per seed {np.round(gain, 3).tolist()} -> "
                   + ('REPLACES ' if win else 'does not replace ') + f"{inc.family}")
        if win: inc = b
    return inc, log

def routing_table(dev, pi, acfg=ACFG):
    rows = []
    for (f, mo), t in dev[dev.eligible].groupby(['family', 'mode']):
        agg = t.groupby(['scope', 'seed']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean')).reset_index()
        gm = {s: agg[agg.scope == s].det.mean() for s in SCOPES}
        bg = 'G0' if gm['G0'] >= gm['G1'] else 'G1'
        dE = agg[agg.scope == 'E'].sort_values('seed').det.values; dG = agg[agg.scope == bg].sort_values('seed').det.values
        aE = agg[agg.scope == 'E'].alarm.mean(); aG = agg[agg.scope == bg].alarm.mean()
        p = pi[(pi.family == f) & (pi['mode'] == mo) & pi.scope.isin(['E', bg])]
        p = p.groupby(['scope', 'mac', 'inst']).hit.mean().reset_index()
        macs = sorted(t.mac.unique()); rng = np.random.default_rng(acfg['boot_seed'])
        H = {m: (p[(p.scope == 'E') & (p.mac == m)].sort_values('inst').hit.values, p[(p.scope == bg) & (p.mac == m)].sort_values('inst').hit.values) for m in macs}
        diffs = []
        for _ in range(acfg['boot_inst']):
            dd = []
            for m in macs:
                a, b = H[m]; i = rng.integers(0, len(a), len(a)); dd.append(a[i].mean() - b[i].mean())
            diffs.append(np.mean(dd))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        passed = bool(all((dE - dG) >= acfg['det_margin']) and lo > 0 and aE <= aG + acfg['alarm_slack'])
        rows.append(dict(family=f, mode=mo, det_E=float(dE.mean()), best_global=bg, det_global=float(dG.mean()), delta_by_seed=np.round(dE - dG, 3).tolist(),
                         ci95=[float(lo), float(hi)], alarm_E=float(aE), alarm_global=float(aG), routing_passes=passed))
    return pd.DataFrame(rows)

def decide(dev, pi, acfg=ACFG):
    ct = cfg_table(dev); sel, log = ladder(ct); rt = routing_table(dev, pi)
    out = dict(config_table=ct, ladder_log=log, routing_table=rt, selected=None if sel is None else sel.to_dict(), routing='no_admissible_detector', effective=None)
    if sel is None: return out
    if sel.scope == 'E':
        r = rt[(rt.family == sel.family) & (rt['mode'] == sel['mode'])].iloc[0]
        if r.routing_passes: out.update(routing='routing_kept', effective=sel.to_dict()); return out
        eff, log2 = ladder(ct, scopes=('G0', 'G1')); out['ladder_log_global_only'] = log2
        out.update(routing='routing_removed_tried_and_failed', effective=None if eff is None else eff.to_dict()); return out
    out.update(routing='routing_not_selected_global_is_best', effective=sel.to_dict()); return out

def detection_tiers(dev, eff, acfg=ACFG):
    if eff is None: return pd.DataFrame([dict(mac=m, device=n, det=np.nan, alarm=np.nan, n_dev_inst=k, detection_supported=False) for m, n, k in
                                         dev[['mac', 'device', 'n_dev_inst']].drop_duplicates().itertuples(index=False)])
    t = dev[(dev.family == eff['family']) & (dev['mode'] == eff['mode']) & (dev.scope == eff['scope'])]
    g = t.groupby(['mac', 'device']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean'), n_dev_inst=('n_dev_inst', 'first')).reset_index()
    g['detection_supported'] = (g.det >= 0.80) & (g.alarm <= 0.02) & (g.n_dev_inst >= acfg['min_dev_inst'])
    return g

# ---------------- identity (development split) ----------------
class MLPArm:
    def __init__(self, seed, clip=20.0): self.seed, self.clip = seed, clip
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed); n = min(3000, max(np.bincount(y)))
        ii = np.concatenate([rng.choice(np.where(y == k)[0], n, replace=True) for k in np.unique(y)])
        self.sc = StandardScaler().fit(X[ii]); Z = np.clip(self.sc.transform(X[ii]), -self.clip, self.clip)
        self.m = MLPClassifier(hidden_layer_sizes=(128, 64), early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                               max_iter=200, random_state=self.seed).fit(Z, y[ii]); return self
    def predict_proba(self, X): return self.m.predict_proba(np.clip(self.sc.transform(X), -self.clip, self.clip))

def id_windows(D):
    W = {}
    for m, d in D.items():
        dtr, dval = d['dev_days']
        for part, days in (('train', dtr), ('calib', d['split']['calib']), ('val', dval)):
            W[(m, part)] = windows(d, days)
            if len(W[(m, part)][0]) == 0: raise ValueError(f"no identity windows for {d['name']} in {part}")
    return W

def run_identity_dev(D, W, cfg, seed, unknown, kinds=('RF', 'XGB', 'MLP')):
    from sklearn.metrics import f1_score
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.utils.class_weight import compute_sample_weight
    known = sorted(m for m in D if m not in unknown); kidx = {m: i for i, m in enumerate(known)}; K = len(known)
    def stack(part, devs):
        Xs, ys, dv, dd = [], [], [], []
        for m in devs:
            F, dy = W[(m, part)]; Xs.append(F); ys += [kidx.get(m, -1)] * len(F); dv += [m] * len(F); dd += list(dy)
        return np.vstack(Xs), np.array(ys), np.array(dv), np.array(dd)
    Xtr, ytr, _, _ = stack('train', known); Xca, yca, vca, _ = stack('calib', known); Xva, yva, vva, dva = stack('val', known)
    Xun, _, vun, _ = stack('val', unknown) if unknown else (np.zeros((0, Xtr.shape[1])), None, np.array([]), None)
    coh = np.array([D[m]['cohort'] for m in known])
    def pdv(P, vv):
        o = P.copy()
        for m in np.unique(vv):
            ii = np.where(vv == m)[0]; o[ii] = vote(P[ii])
        return o
    res = {}
    for kind in kinds:
        if kind == 'RF': mdl = RandomForestClassifier(n_estimators=cfg['rf_trees'], class_weight='balanced', n_jobs=-1, random_state=seed).fit(Xtr, ytr)
        elif kind == 'XGB':
            from xgboost import XGBClassifier
            mdl = XGBClassifier(n_estimators=cfg['xgb_trees'], max_depth=6, learning_rate=0.05, objective='multi:softprob', tree_method='hist',
                                n_jobs=-1, random_state=seed, num_class=K).fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
        else: mdl = MLPArm(seed).fit(Xtr, ytr)
        P = dict(ca=mdl.predict_proba(Xca), va=mdl.predict_proba(Xva), un=mdl.predict_proba(Xun) if len(Xun) else np.zeros((0, K)))
        for mode in ('single', 'vote3'):
            Q = P if mode == 'single' else dict(ca=pdv(P['ca'], vca), va=pdv(P['va'], vva), un=pdv(P['un'], vun) if len(Xun) else P['un'])
            thr = np.array([np.quantile(Q['ca'][yca == k, k], 0.10) for k in range(K)])
            pred = Q['va'].argmax(1); f1_dev = f1_score(yva, pred, average=None, labels=list(range(K)))
            acc = (Q['va'][np.arange(len(yva)), yva] >= thr[yva]).astype(float)
            rng = np.random.default_rng(seed)
            claim = np.array([rng.choice([k for k in range(K) if k != y]) for y in yva])
            rej = (Q['va'][np.arange(len(yva)), claim] < thr[claim]).astype(float)
            same = coh[claim] == coh[yva]
            hard = {}
            for A in range(K):
                mp = {B: Q['ca'][yca == B][:, A].mean() for B in range(K) if B != A}; hard[A] = max(mp, key=mp.get)
            rh = np.array([float(Q['va'][i, A] < thr[A]) for A in range(K) for i in np.where(yva == hard[A])[0]])
            ru = (Q['un'][:, :] < thr[None, :]).astype(float).ravel() if len(Xun) else np.array([])
            allrej = np.concatenate([rej, rh, ru])
            res[f'{kind}-{mode}'] = dict(
                f1_calib=float(f1_score(yca, Q['ca'].argmax(1), average='macro')), G1=float(f1_score(yva, pred, average='macro')),
                G2=float(np.mean([acc[yva == k].mean() for k in range(K)])), G3=float(rej.mean()), G4=float(rh.mean()),
                G5=float(ru.mean()) if len(ru) else None, G6=float(1 - allrej.mean()),
                rej_same_cohort=float(rej[same].mean()) if same.any() else None, rej_cross_cohort=float(rej[~same].mean()) if (~same).any() else None,
                per_device={known[k]: dict(f1=float(f1_dev[k]), acceptance=float(acc[yva == k].mean()), claim_rejection=float(rej[claim == k].mean()) if (claim == k).any() else None)
                            for k in range(K)}, n_val=int(len(yva)))
    return res

def identity_summary(held, closed, seeds, tol=0.0):
    """held/closed: {seed: {arm: metrics}}. Returns admission + primary arm + per-device decision (closed-set, primary arm, seed means)."""
    arms = sorted(held[seeds[0]])
    g2 = {a: [held[s][a]['G2'] for s in seeds] for a in arms}; g6 = {a: [held[s][a]['G6'] for s in seeds] for a in arms}
    simple = [a for a in arms if not a.startswith('MLP')]; mlp = [a for a in arms if a.startswith('MLP')]
    admitted, note = list(simple), []
    if mlp:
        for a in mlp:
            mode = a.split('-')[1]; ref = [x for x in simple if x.endswith(mode)]
            best = max(ref, key=lambda x: np.mean(g2[x]))
            ok = all(g2[a][i] - g2[best][i] >= 0.05 for i in range(len(seeds))) and all(g6[a][i] - g6[best][i] <= 0.01 for i in range(len(seeds)))
            note.append(f'{a} vs {best}: G2 gains per seed {np.round(np.array(g2[a]) - np.array(g2[best]), 3).tolist()} -> ' + ('ADMITTED' if ok else 'not admitted'))
            if ok: admitted.append(a)
    primary = max(admitted, key=lambda a: (round(np.mean([held[s][a]['f1_calib'] for s in seeds]), 12), -arms.index(a)))
    devs = closed[seeds[0]][primary]['per_device']; dec = []
    for m in devs:
        f1 = np.mean([closed[s][primary]['per_device'][m]['f1'] for s in seeds]); ac = np.mean([closed[s][primary]['per_device'][m]['acceptance'] for s in seeds])
        cr = [closed[s][primary]['per_device'][m]['claim_rejection'] for s in seeds if closed[s][primary]['per_device'][m]['claim_rejection'] is not None]
        crm = float(np.mean(cr)) if cr else np.nan
        dec.append(dict(mac=m, f1=float(f1), acceptance=float(ac), claim_rejection=crm, identity_supported=bool(f1 >= 0.80 and ac >= 0.80 and crm >= 0.95)))
    return dict(primary=primary, admitted=admitted, mlp_admission=note, device_identity=pd.DataFrame(dec),
                held_gates={a: {k: float(np.mean([held[s][a][k] for s in seeds])) for k in ('G1', 'G2', 'G3', 'G4', 'G5', 'G6')} for a in arms})

def assign_tiers(D, ident, dtier):
    t = ident['device_identity'].merge(dtier[['mac', 'det', 'alarm', 'n_dev_inst', 'detection_supported']], on='mac', how='left')
    t['device'] = t.mac.map({m: D[m]['name'] for m in D})
    def tier(r):
        if not r.identity_supported: return 'UNREGISTERED'
        return 'VERIFIED' if bool(r.detection_supported) else 'IDENTIFIED_UNVALIDATED'
    t['tier'] = t.apply(tier, axis=1)
    return t[['device', 'mac', 'tier', 'f1', 'acceptance', 'claim_rejection', 'identity_supported', 'det', 'alarm', 'n_dev_inst', 'detection_supported']]

In [ ]:
"""N2 Phase A, iteration 2. Implements PROTOCOL_A_ITER2.md. Needs n2_lib.py and phaseA_lib.py (pasted above it in the notebook).
Still uses ODD instances and the development split only."""
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
try:
    from n2_lib import attack_parts, windows, vote
    from phaseA_lib import ACFG, AE, lagstack, hits_arr, prepare_dev, id_windows, MLPArm
except ImportError:
    pass

A2 = dict(modes=('static', 'rollq', 'both'), roll_q=0.99, roll_win=4320, roll_min=1440, block=10, det_min=0.80, alarm_max=0.02,
          oof_folds=5, oof_trees=300, id_arms=('RF-single', 'RF-vote3'), rules=('T0', 'T1'))

# ---------------- detection ----------------
def score_ae(D, seed, acfg, cfg):
    """Train AE expert (E), global (G0) and global+one-hot (G1); return per-minute scores for every device."""
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}; eye = np.eye(len(macs), dtype=np.float32)
    rng = np.random.default_rng(seed)
    new = lambda: AE(acfg['ae_hidden'], acfg['ae_epochs'], seed, acfg['clip'])
    def samp(m, n):
        ii = np.where(D[m]['m_dtrain'])[0]
        return np.sort(rng.choice(ii, n, replace=False)) if len(ii) > n else ii
    X = {m: D[m]['F'] for m in macs}
    E = {m: new().fit(X[m][samp(m, acfg['cap_dev'])]) for m in macs}
    per = acfg['cap_pool'] // len(macs); Xp = {m: X[m][samp(m, per)] for m in macs}
    G0 = new().fit(np.vstack([Xp[m] for m in macs]))
    G1 = new().fit(np.vstack([np.hstack([Xp[m], np.tile(eye[idx[m]], (len(Xp[m]), 1))]) for m in macs]))
    S = {'E': {}, 'G0': {}, 'G1': {}}
    for m in macs:
        S['E'][m] = E[m].score(X[m]).astype(np.float32); S['G0'][m] = G0.score(X[m]).astype(np.float32)
        S['G1'][m] = G1.score(np.hstack([X[m], np.tile(eye[idx[m]], (len(X[m]), 1))])).astype(np.float32)
    return S

def rollq_thr(s, static_thr, a2=A2):
    t = pd.Series(s).rolling(a2['roll_win'], min_periods=a2['roll_min']).quantile(a2['roll_q']).shift(1).to_numpy()
    return np.where(np.isnan(t), static_thr, t)

def block_rate(alarm, mask, size):
    ii = np.where(mask)[0]; n = len(ii) // size
    if n == 0: return np.nan
    return float((alarm[ii[:n * size]].reshape(n, size).sum(1) >= 2).mean())

def eval_modes(D, S, seed, a2=A2):
    macs = list(D); thr0 = np.quantile(np.concatenate([S['G0'][m][D[m]['m_calib']] for m in macs]), 0.99)
    rows, prow = [], []
    for scope in ('E', 'G0', 'G1'):
        for m in macs:
            d = D[m]; s = S[scope][m].astype(float)
            st = thr0 if scope == 'G0' else float(np.quantile(s[d['m_calib']], 0.99))
            al_s = s >= st; al_r = s >= rollq_thr(s, st, a2)
            for mode, al in (('static', al_s), ('rollq', al_r), ('both', al_s & al_r)):
                h = hits_arr(d['ts'], al, d['dev_inst'])
                rows.append(dict(family='AE', mode=mode, scope=scope, seed=seed, mac=m, device=d['name'], eligible=d['eligible'],
                                 n_dev_inst=len(d['dev_inst']), det=float(h.mean()), alarm_val=float(al[d['m_dval']].mean()),
                                 alarm_block=block_rate(al, d['m_dval'], a2['block']), alarm_calib=float(al[d['m_calib']].mean())))
                for j, v in enumerate(h): prow.append(('AE', mode, scope, seed, m, j, int(v)))
    return pd.DataFrame(rows), pd.DataFrame(prow, columns=['family', 'mode', 'scope', 'seed', 'mac', 'inst', 'hit'])

# ---------------- selection by number of supported devices ----------------
def support_table(dev, a2=A2):
    el = dev[dev.eligible]
    g = el.groupby(['mode', 'scope', 'mac', 'device']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean'), block=('alarm_block', 'mean'),
                                                          n_dev_inst=('n_dev_inst', 'first')).reset_index()
    g['supported'] = (g.det >= a2['det_min']) & (g.alarm <= a2['alarm_max'])
    rows = []
    for (mo, sc), t in g.groupby(['mode', 'scope']):
        s = t[t.supported]
        rows.append(dict(mode=mo, scope=sc, n_supported=int(len(s)), supported=sorted(s.device), det_supported=float(s.det.mean()) if len(s) else 0.0,
                         det_all=float(t.det.mean()), alarm_all=float(t.alarm.mean()), alarm_block_all=float(t.block.mean()),
                         worst_alarm_device=t.sort_values('alarm').iloc[-1].device, worst_alarm=float(t.alarm.max())))
    return g, pd.DataFrame(rows)

def select2(st, scopes=('E', 'G0', 'G1')):
    t = st[st.scope.isin(scopes)].copy()
    t['mr'] = t['mode'].map({'static': 0, 'rollq': 1, 'both': 2}); t['sr'] = t.scope.map({'G0': 0, 'G1': 1, 'E': 2}); t['d'] = t.det_supported.round(12)
    return t.sort_values(['n_supported', 'd', 'mr', 'sr'], ascending=[False, False, True, True]).iloc[0]

def routing2(dev, pi, g, st, a2=A2, acfg=ACFG):
    rows = []
    for mo in a2['modes']:
        bg = select2(st[st['mode'] == mo], ('G0', 'G1')).scope
        ge, gg = g[(g['mode'] == mo) & (g.scope == 'E')], g[(g['mode'] == mo) & (g.scope == bg)]
        common = sorted(set(ge[ge.alarm <= a2['alarm_max']].mac) & set(gg[gg.alarm <= a2['alarm_max']].mac))
        if not common:
            rows.append(dict(mode=mo, best_global=bg, n_common=0, routing_passes=False)); continue
        d = dev[(dev['mode'] == mo) & dev.mac.isin(common) & dev.scope.isin(['E', bg])]
        dE = d[d.scope == 'E'].groupby('seed').det.mean().sort_index().values; dG = d[d.scope == bg].groupby('seed').det.mean().sort_index().values
        p = pi[(pi['mode'] == mo) & pi.mac.isin(common) & pi.scope.isin(['E', bg])].groupby(['scope', 'mac', 'inst']).hit.mean().reset_index()
        rng = np.random.default_rng(acfg['boot_seed']); H = {m: (p[(p.scope == 'E') & (p.mac == m)].sort_values('inst').hit.values,
                                                                  p[(p.scope == bg) & (p.mac == m)].sort_values('inst').hit.values) for m in common}
        diffs = []
        for _ in range(acfg['boot_inst']):
            dd = []
            for m in common:
                a, b = H[m]; i = rng.integers(0, len(a), len(a)); dd.append(a[i].mean() - b[i].mean())
            diffs.append(np.mean(dd))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        rows.append(dict(mode=mo, best_global=bg, n_common=len(common), common=[D_ for D_ in d.device.unique()], det_E=float(dE.mean()), det_global=float(dG.mean()),
                         delta_by_seed=np.round(dE - dG, 3).tolist(), ci95=[float(lo), float(hi)],
                         routing_passes=bool(all((dE - dG) >= acfg['det_margin']) and lo > 0)))
    return pd.DataFrame(rows)

def decide2(dev, pi, a2=A2):
    g, st = support_table(dev, a2); sel = select2(st); rt = routing2(dev, pi, g, st, a2)
    out = dict(support=st, per_device=g, routing_table=rt, selected=sel.to_dict(), routing='', effective=None)
    if sel.scope == 'E':
        if bool(rt[rt['mode'] == sel['mode']].iloc[0].routing_passes): out.update(routing='routing_kept', effective=sel.to_dict())
        else: out.update(routing='routing_removed_tried_and_failed', effective=select2(st, ('G0', 'G1')).to_dict())
    else: out.update(routing='routing_not_selected_global_is_best', effective=sel.to_dict())
    return out

def detection_tiers2(g, eff, D):
    t = g[(g['mode'] == eff['mode']) & (g.scope == eff['scope'])].copy()
    t['detection_supported'] = t.supported & (t.n_dev_inst >= ACFG['min_dev_inst'])
    return t.rename(columns={'alarm': 'alarm'})[['mac', 'device', 'det', 'alarm', 'block', 'n_dev_inst', 'detection_supported']]

# ---------------- identity with out-of-fold thresholds ----------------
def _metrics(Q, thr, yca, yva, vva, claim, coh, hard_rows, Qun, K, f1fn):
    pred = Q['va'].argmax(1); f1_dev = f1fn(yva, pred, average=None, labels=list(range(K)))
    acc = (Q['va'][np.arange(len(yva)), yva] >= thr[yva]).astype(float)
    rej = (Q['va'][np.arange(len(yva)), claim] < thr[claim]).astype(float); same = coh[claim] == coh[yva]
    rh = np.array([float(Q['va'][i, A] < thr[A]) for i, A in hard_rows])
    ru = (Qun < thr[None, :]).astype(float).ravel() if len(Qun) else np.array([])
    allrej = np.concatenate([rej, rh, ru])
    return dict(G1=float(f1fn(yva, pred, average='macro')), G2=float(np.mean([acc[yva == k].mean() for k in range(K)])), G3=float(rej.mean()),
                G4=float(rh.mean()), G5=float(ru.mean()) if len(ru) else None, G6=float(1 - allrej.mean()),
                rej_same_cohort=float(rej[same].mean()) if same.any() else None, rej_cross_cohort=float(rej[~same].mean()) if (~same).any() else None,
                per_device={k: dict(f1=float(f1_dev[k]), acceptance=float(acc[yva == k].mean()),
                                    claim_rejection=float(rej[claim == k].mean()) if (claim == k).any() else None) for k in range(K)})

def run_identity2(D, W, cfg, seed, unknown, a2=A2):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import f1_score
    known = sorted(m for m in D if m not in unknown); kidx = {m: i for i, m in enumerate(known)}; K = len(known)
    def stack(part, devs):
        Xs, ys, dv, dd = [], [], [], []
        for m in devs:
            F, dy = W[(m, part)]; Xs.append(F); ys += [kidx.get(m, -1)] * len(F); dv += [m] * len(F); dd += [f'{m}_{x}' for x in dy]
        return np.vstack(Xs), np.array(ys), np.array(dv), np.array(dd)
    Xtr, ytr, vtr, gtr = stack('train', known); Xca, yca, vca, _ = stack('calib', known); Xva, yva, vva, _ = stack('val', known)
    Xun, _, vun, _ = stack('val', unknown) if unknown else (np.zeros((0, Xtr.shape[1])), None, np.array([]), None)
    coh = np.array([D[m]['cohort'] for m in known])
    rf = lambda n: RandomForestClassifier(n_estimators=n, class_weight='balanced', n_jobs=-1, random_state=seed)
    mdl = rf(cfg['rf_trees']).fit(Xtr, ytr)
    P = dict(ca=mdl.predict_proba(Xca), va=mdl.predict_proba(Xva), un=mdl.predict_proba(Xun) if len(Xun) else np.zeros((0, K)))
    oof = np.zeros((len(Xtr), K))                                  # day-grouped out-of-fold probabilities on TRAIN windows
    for a, b in GroupKFold(a2['oof_folds']).split(Xtr, ytr, gtr):
        f = rf(a2['oof_trees']).fit(Xtr[a], ytr[a]); pp = f.predict_proba(Xtr[b]); oof[np.ix_(b, f.classes_)] = pp
    def pdv(Pm, vv):
        o = Pm.copy()
        for m in np.unique(vv):
            ii = np.where(vv == m)[0]; o[ii] = vote(Pm[ii])
        return o
    rng = np.random.default_rng(seed); claim = np.array([rng.choice([k for k in range(K) if k != y]) for y in yva])
    res, diag = {}, {}
    for mode in ('single', 'vote3'):
        Q = P if mode == 'single' else dict(ca=pdv(P['ca'], vca), va=pdv(P['va'], vva), un=pdv(P['un'], vun) if len(Xun) else P['un'])
        Qo = oof if mode == 'single' else pdv(oof, vtr)
        thr0 = np.array([np.quantile(Q['ca'][yca == k, k], 0.10) for k in range(K)])
        thr1 = np.array([np.quantile(np.concatenate([Q['ca'][yca == k, k], Qo[ytr == k, k]]), 0.10) for k in range(K)])
        hard = {}
        for A in range(K):
            mp = {B: Q['ca'][yca == B][:, A].mean() for B in range(K) if B != A}; hard[A] = max(mp, key=mp.get)
        hard_rows = [(i, A) for A in range(K) for i in np.where(yva == hard[A])[0]]
        for rule, thr in (('T0', thr0), ('T1', thr1)):
            r = _metrics(Q, thr, yca, yva, vva, claim, coh, hard_rows, Q['un'], K, f1_score)
            r['f1_calib'] = float(f1_score(yca, Q['ca'].argmax(1), average='macro'))
            r['per_device'] = {known[k]: v for k, v in r['per_device'].items()}
            res[f'RF-{mode}|{rule}'] = r
        diag[mode] = {known[k]: dict(thr0=float(thr0[k]), thr1=float(thr1[k]), p_oof=float(np.median(Qo[ytr == k, k])), p_calib=float(np.median(Q['ca'][yca == k, k])),
                                      p_val=float(np.median(Q['va'][yva == k, k])), acc_val_T0=float((Q['va'][yva == k, k] >= thr0[k]).mean()),
                                      acc_val_T1=float((Q['va'][yva == k, k] >= thr1[k]).mean())) for k in range(K)}
    return res, diag

def identity_summary2(held, closed, seeds, a2=A2):
    arms = [f'{a}|{r}' for a in a2['id_arms'] for r in a2['rules']]
    primary_arm = max(a2['id_arms'], key=lambda a: (round(np.mean([held[s][f'{a}|T0']['f1_calib'] for s in seeds]), 12), -a2['id_arms'].index(a)))
    hg = {x: {k: float(np.mean([held[s][x][k] for s in seeds])) for k in ('G1', 'G2', 'G3', 'G4', 'G5', 'G6')} for x in arms}
    def dec(rule):
        x = f'{primary_arm}|{rule}'; rows = []
        for m in closed[seeds[0]][x]['per_device']:
            f1 = np.mean([closed[s][x]['per_device'][m]['f1'] for s in seeds]); ac = np.mean([closed[s][x]['per_device'][m]['acceptance'] for s in seeds])
            cr = np.mean([closed[s][x]['per_device'][m]['claim_rejection'] for s in seeds])
            rows.append(dict(mac=m, f1=float(f1), acceptance=float(ac), claim_rejection=float(cr), identity_supported=bool(f1 >= 0.80 and ac >= 0.80 and cr >= 0.95)))
        return pd.DataFrame(rows)
    d0, d1 = dec('T0'), dec('T1'); g0, g1 = hg[f'{primary_arm}|T0'], hg[f'{primary_arm}|T1']
    gates_ok = g1['G2'] >= 0.80 and g1['G3'] >= 0.95 and g1['G4'] >= 0.90 and g1['G5'] >= 0.60 and g1['G6'] <= 0.10
    adopt = bool(gates_ok and d1.identity_supported.sum() > d0.identity_supported.sum() and g1['G6'] <= g0['G6'] + 0.01)
    return dict(primary_arm=primary_arm, held_gates=hg, device_T0=d0, device_T1=d1, adopt_T1=adopt, rule='T1' if adopt else 'T0',
                device_identity=d1 if adopt else d0)

def assign_tiers2(D, ident, dtier):
    t = ident['device_identity'].merge(dtier[['mac', 'det', 'alarm', 'block', 'n_dev_inst', 'detection_supported']], on='mac', how='left')
    t['device'] = t.mac.map({m: D[m]['name'] for m in D})
    t['detection_supported'] = t.detection_supported.fillna(False).astype(bool)
    t['tier'] = t.apply(lambda r: 'UNREGISTERED' if not r.identity_supported else ('VERIFIED' if bool(r.detection_supported) else 'IDENTIFIED_UNVALIDATED'), axis=1)
    return t[['device', 'mac', 'tier', 'f1', 'acceptance', 'claim_rejection', 'identity_supported', 'det', 'alarm', 'block', 'n_dev_inst', 'detection_supported']]

## Download data

In [ ]:
import os, zipfile, shutil, requests, glob, hashlib, time
OUT = '/kaggle/working/n2_phaseA_iter2'; os.makedirs(OUT, exist_ok=True)
DATA = '/kaggle/working/unsw_attack'; os.makedirs(DATA, exist_ok=True)
BASE = 'https://iotanalytics.unsw.edu.au/anomaly-data/'
hashes = {}
for n in ['flowdata.zip', 'annotations.zip']:
    p = (glob.glob(f'/kaggle/input/**/{n}', recursive=True) or [None])[0]
    if p is None:
        p = f'{DATA}/{n}'
        with requests.get(BASE + n, stream=True, timeout=600, headers={'User-Agent': 'ris-n2'}) as r:
            r.raise_for_status()
            with open(p, 'wb') as fh: shutil.copyfileobj(r.raw, fh)
    hashes[n] = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    with zipfile.ZipFile(p) as z: z.extractall(f'{DATA}/{n[:-4]}')
print(hashes); json.dump(hashes, open(f'{OUT}/data_hashes.json', 'w'), indent=1)

## Load and development split

In [ ]:
D = load_all(f'{DATA}/flowdata', f'{DATA}/annotations'); prepare_dev(D)
assert all('m_test' not in d for d in D.values())
SEEDS = ACFG['seeds']; print('DEV instances:', {d['name']: len(d['dev_inst']) for d in D.values()})

## I1 Identity: RF single/vote3, threshold rules T0 and T1, diagnostics

In [ ]:
t0 = time.time(); W = id_windows(D); unk = pick_unknown(D, CFG['split_seed'])
print('held-out identity devices:', [D[m]['name'] for m in unk])
held, closed, diag = {}, {}, {}
for s in SEEDS:
    held[s], _ = run_identity2(D, W, CFG, s, unk); closed[s], diag[s] = run_identity2(D, W, CFG, s, [])
isum = identity_summary2(held, closed, SEEDS)
print('primary arm:', isum['primary_arm'], '| adopt T1:', isum['adopt_T1'], '| rule used:', isum['rule'])
print(pd.DataFrame(isum['held_gates']).T.round(3).to_string())
nm = {m: D[m]['name'] for m in D}
print('closed-set per device, T0:'); print(isum['device_T0'].assign(device=lambda t: t.mac.map(nm)).round(3).to_string(index=False))
print('closed-set per device, T1:'); print(isum['device_T1'].assign(device=lambda t: t.mac.map(nm)).round(3).to_string(index=False))
for mode in ('single', 'vote3'):
    print('diagnostics', mode, '(seed 11):'); print(pd.DataFrame(diag[11][mode]).T.rename(index=nm).round(3).to_string())
print('same/cross-cohort mismatch rejection (T0, seed 11):', held[11][isum['primary_arm'] + '|T0']['rej_same_cohort'], held[11][isum['primary_arm'] + '|T0']['rej_cross_cohort'])
json.dump(dict(held=held, closed=closed, diag=diag), open(f'{OUT}/identity2_runs.json', 'w'), indent=1, default=float)
isum['device_T0'].to_csv(f'{OUT}/identity2_device_T0.csv', index=False); isum['device_T1'].to_csv(f'{OUT}/identity2_device_T1.csv', index=False)
print('identity done', round(time.time() - t0), 's')

## D1-D3 Detection: AE x modes (static, rollq, both) x scopes x seeds

In [ ]:
devs, pis = [], []
for s in SEEDS:
    fp = f'{OUT}/det2_AE_{s}.csv'
    if os.path.exists(fp) and os.path.exists(fp.replace('det2_', 'pi2_')):
        devs.append(pd.read_csv(fp)); pis.append(pd.read_csv(fp.replace('det2_', 'pi2_'))); continue
    t0 = time.time(); S = score_ae(D, s, ACFG, CFG)
    np.savez_compressed(f'{OUT}/scores_AE_{s}.npz', **{f'{sc}_{m}': S[sc][m] for sc in S for m in S[sc]})
    a, b = eval_modes(D, S, s); a.to_csv(fp, index=False); b.to_csv(fp.replace('det2_', 'pi2_'), index=False); devs.append(a); pis.append(b)
    print('AE seed', s, round(time.time() - t0), 's', flush=True)
dev = pd.concat(devs, ignore_index=True); pi = pd.concat(pis, ignore_index=True)
dev.to_csv(f'{OUT}/detection2_dev_all.csv', index=False); pi.to_csv(f'{OUT}/detection2_instances_all.csv', index=False)

## Decision: supported-device selection, routing test, tiers

In [ ]:
out = decide2(dev, pi)
st = out['support']; print(st.drop(columns=['supported']).round(4).to_string(index=False))
print('supported devices per configuration:'); print(st[['mode', 'scope', 'supported']].to_string(index=False))
print(out['routing_table'].round(3).to_string(index=False))
print('SELECTED:', out['selected']['mode'], out['selected']['scope'], '| ROUTING:', out['routing'], '| EFFECTIVE:', out['effective']['mode'], out['effective']['scope'])
print(out['per_device'][(out['per_device']['mode'] == out['effective']['mode']) & (out['per_device'].scope == out['effective']['scope'])].round(3).to_string(index=False))
print('AE static: per device alarm (E / G0 / G1):'); print(out['per_device'][out['per_device']['mode'] == 'static'].pivot(index='device', columns='scope', values='alarm').round(3).to_string())
dtier = detection_tiers2(out['per_device'], out['effective'], D); tiers = assign_tiers2(D, isum, dtier)
print(tiers.round(3).to_string(index=False)); tiers.to_csv(f'{OUT}/device_tiers_iter2.csv', index=False)
st.drop(columns=['supported']).to_csv(f'{OUT}/support_table.csv', index=False); out['routing_table'].to_csv(f'{OUT}/routing_table2.csv', index=False)

## Save

In [ ]:
final = dict(protocol_sha256=sha, chain_block=blk['hash'], data_hashes=hashes, a2={k: (list(v) if isinstance(v, tuple) else v) for k, v in A2.items()},
             identity=dict(primary_arm=isum['primary_arm'], rule=isum['rule'], adopt_T1=isum['adopt_T1'], held_gates=isum['held_gates']),
             detection=dict(selected=out['selected'], effective=out['effective'], routing=out['routing']), tiers=tiers.to_dict('records'))
json.dump(final, open(f'{OUT}/phaseA_iter2_results.json', 'w'), indent=1, default=str)
shutil.make_archive('/kaggle/working/n2_phaseA_iter2_results', 'zip', OUT)
print('TIERS:', tiers.groupby('tier').device.apply(list).to_dict()); print('ROUTING:', out['routing'], '| IDENTITY RULE:', isum['rule'], '| PRIMARY ARM:', isum['primary_arm'])